In [1]:
import os
import pandas as pd
from pathlib import Path
from datetime import datetime

def scan_project_directory(root_dir):
    records = []
    root_path = Path(root_dir).resolve()
    
    for dirpath, _, filenames in os.walk(root_path):
        for filename in filenames:
            try:
                filepath = Path(dirpath) / filename
                stat = filepath.stat()
                records.append({
                    "relative_path": str(filepath.relative_to(root_path)),
                    "absolute_path": str(filepath),
                    "name": filename,
                    "extension": filepath.suffix.lower(),
                    "size_MB": round(stat.st_size / (1024 * 1024), 3),
                    "last_modified": datetime.fromtimestamp(stat.st_mtime),
                    "last_accessed": datetime.fromtimestamp(stat.st_atime),
                    "created": datetime.fromtimestamp(stat.st_ctime),
                })
            except Exception as e:
                print(f"⚠️ Error reading {filepath}: {e}")

    df = pd.DataFrame(records)
    return df.sort_values(by="size_MB", ascending=False).reset_index(drop=True)

# 🔍 Scan current project directory (or replace '.' with a specific path)
file_df = scan_project_directory(".")
file_df.head(20)  # Show top 20 largest files


,relative_path,absolute_path,name,extension,size_MB,last_modified,last_accessed,created
0,Downloads/wormbase_full_release/hub/ws268.hal,/home/rohit/Desktop/Projects/C-Elegans/Downloa...,ws268.hal,.hal,2454.646,2025-05-27 06:45:00.000000,2025-08-07 07:49:19.597274,2025-08-07 06:26:01.908499
1,Downloads/wormbase_full_release/database.297.4...,/home/rohit/Desktop/Projects/C-Elegans/Downloa...,database.297.4-16.tar.gz,.gz,1201.947,2025-05-27 06:42:00.000000,2025-08-08 11:16:16.283029,2025-08-07 06:25:23.231679
2,Downloads/wormbase_full_release/database.297.4...,/home/rohit/Desktop/Projects/C-Elegans/Downloa...,database.297.4-2.tar.gz,.gz,990.819,2025-05-27 06:42:00.000000,2025-08-08 11:16:16.931042,2025-08-07 06:22:36.856143
3,_extracted_tars/database.297.4-3/database/bloc...,/home/rohit/Desktop/Projects/C-Elegans/_extrac...,block13.wrm,.wrm,976.500,2025-05-19 23:27:04.000000,2025-05-19 23:27:04.000000,2025-08-07 13:48:52.375194
4,_extracted_tars/database.297.4-5/database/bloc...,/home/rohit/Desktop/Projects/C-Elegans/_extrac...,block24.wrm,.wrm,976.500,2025-05-19 23:25:57.000000,2025-05-19 23:25:57.000000,2025-08-07 13:53:57.821479
5,_extracted_tars/database.297.4-5/database/bloc...,/home/rohit/Desktop/Projects/C-Elegans/_extrac...,block23.wrm,.wrm,976.500,2025-05-19 23:25:57.000000,2025-05-19 23:25:57.000000,2025-08-07 13:53:27.584856
6,_extracted_tars/database.297.4-15/database/blo...,/home/rohit/Desktop/Projects/C-Elegans/_extrac...,block74.wrm,.wrm,976.500,2025-05-19 23:26:58.000000,2025-05-19 23:26:58.000000,2025-08-07 14:21:52.294432
7,_extracted_tars/database.297.4-15/database/blo...,/home/rohit/Desktop/Projects/C-Elegans/_extrac...,block75.wrm,.wrm,976.500,2025-05-19 23:27:03.000000,2025-05-19 23:27:03.000000,2025-08-07 14:22:09.934935
8,_extracted_tars/database.297.4-15/database/blo...,/home/rohit/Desktop/Projects/C-Elegans/_extrac...,block73.wrm,.wrm,976.500,2025-05-19 12:16:32.000000,2025-05-19 12:16:32.000000,2025-08-07 14:21:33.761939
9,_extracted_tars/database.297.4-15/database/blo...,/home/rohit/Desktop/Projects/C-Elegans/_extrac...,block71.wrm,.wrm,976.500,2025-05-17 07:18:52.000000,2025-05-17 07:18:52.000000,2025-08-07 14:21:25.993747


In [2]:
from collections import defaultdict

def count_files_by_directory(root_dir="."):
    dir_counts = defaultdict(int)
    for dirpath, _, filenames in os.walk(root_dir):
        rel_path = os.path.relpath(dirpath, root_dir)
        file_count = len(filenames)
        dir_counts[rel_path] += file_count
    return pd.DataFrame([
        {"folder": k, "file_count": v}
        for k, v in sorted(dir_counts.items(), key=lambda x: -x[1])
    ])

dir_summary = count_files_by_directory(".")
dir_summary.head(20)  # Show top 20 most crowded folders


,folder,file_count
0,data/temp,877
1,data/neuroml/cell_library,713
2,_extracted_txt/Ce_synapse_UNIX/Ce_synapse/Whit...,211
3,_extracted_tars/database.297.4-0/wquery,138
4,.,65
5,Downloads/wormbase_full_release,53
6,Downloads/wormbase_full_release/hub/remanei,49
7,Downloads/wormbase_full_release/hub/elegans,45
8,Downloads/wormbase_full_release/hub/brenneri,44
9,Downloads/wormbase_full_release/hub/japonica,44


In [3]:
# Count file types (extensions) overall
ext_counts = file_df["extension"].value_counts().reset_index()
ext_counts.columns = ["extension", "count"]
ext_counts.head(20)


,extension,count
0,,828
1,.xml,703
2,.nml,691
3,.gz,538
4,.txt,310
5,.bb,240
6,.def,198
7,.html,109
8,.wrm,94
9,.html?c=d;o=a,86


In [7]:
import pandas as pd
from pathlib import Path

# 🧠 Function to scan all .csv files and extract column names
def list_csv_columns(root_dir="."):
    csv_info = []
    for path in Path(root_dir).rglob("*.csv"):
        try:
            df = pd.read_csv(path, nrows=0)  # Only reads header
            csv_info.append({
                "file": str(path.relative_to(root_dir)),
                "columns": list(df.columns)
            })
        except Exception as e:
            csv_info.append({
                "file": str(path.relative_to(root_dir)),
                "columns": f"⚠️ Error: {e}"
            })
    return pd.DataFrame(csv_info)

# 📦 Run the scan
csv_columns_df = list_csv_columns(".")

# 🧾 Configure pandas to show all rows and columns
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", 200)

# 📊 Display the results
display(csv_columns_df)

# 💾 Optionally save to file
csv_columns_df.to_csv("csv_column_index.csv", index=False)
print("✅ Saved index as csv_column_index.csv")


,file,columns
0,node2vec_embeddings.csv,"[Neuron, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 4..."
1,nml_results.csv,"[FilePath, Tag, Attributes, Text, Pre, Post, Neuron, Source, Target]"
2,strength_metrics.csv,"[Neuron, TotalStrength, AvgRelatedness, StrengthImbalance]"
3,symbolic_clusters_labeled_canonical.csv,"[Neuron, Cluster, TSNE_1, TSNE_2, NeuronName, Canonical]"
4,ExtraData_Sheet_Classification.csv,"[Sheet, Rows, Cols, Columns, Classification]"
5,parsed_txt_salvaged.csv,"[**************************************************************************, __SourceFile, ADAL, Unnamed: 1, Unnamed: 2, Unnamed: 3, ADA, Unnamed: 5, Unnamed: 6, Unnamed: 7, in, I1, A 1 ..."
6,neuron_master.csv,"[Neuron, Type, TotalStrength, AvgRelatedness, StrengthImbalance, RichClubScore, total_3node_motifs, as_source, as_target, KMeansCluster, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 1..."
7,motif_participation_profiles.csv,"[Neuron, (0, 1)_(0, 2)_(1, 2)_nan_nan_nan_nan_nan_nan, (0, 1)_(0, 2)_nan_nan_nan_nan_nan_nan_nan, (0, 1)_(2, 1)_nan_nan_nan_nan_nan_nan_nan, (0, 1)_(0, 2)_(2, 2)_nan_nan_nan_nan_nan_nan, (0, 1)_(0..."
8,latent_archetype_labels.csv,"[Unnamed: 0, AutoLabel]"
9,nca_symbolic_clusters.csv,"[Neuron, Cluster, TSNE_1, TSNE_2]"


✅ Saved index as csv_column_index.csv


In [9]:
import pandas as pd

# Load uploaded CSV column index
csv_index = pd.read_csv("csv_column_index.csv")

# Convert stringified column lists back to Python lists if needed
import ast

def safe_parse_columns(x):
    try:
        parsed = ast.literal_eval(x)
        return parsed if isinstance(parsed, list) else [x]
    except Exception:
        return [x]

csv_index["columns"] = csv_index["columns"].apply(safe_parse_columns)
csv_index.head()


,file,columns
0,node2vec_embeddings.csv,"[Neuron, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 4..."
1,nml_results.csv,"[FilePath, Tag, Attributes, Text, Pre, Post, Neuron, Source, Target]"
2,strength_metrics.csv,"[Neuron, TotalStrength, AvgRelatedness, StrengthImbalance]"
3,symbolic_clusters_labeled_canonical.csv,"[Neuron, Cluster, TSNE_1, TSNE_2, NeuronName, Canonical]"
4,ExtraData_Sheet_Classification.csv,"[Sheet, Rows, Cols, Columns, Classification]"


In [10]:
# Create a stringified column signature
csv_index["schema_signature"] = csv_index["columns"].apply(lambda cols: str(sorted(cols)))

# Group by column signature
grouped = csv_index.groupby("schema_signature").agg({
    "file": list,
    "schema_signature": "count"
}).rename(columns={"schema_signature": "file_count"}).reset_index()

# Sort by most common schema
grouped.sort_values("file_count", ascending=False).head(10)


,schema_signature,file,file_count
66,['pre\tpost\ttype\tsynapses'],"[aconnectome_white_1986_whole.csv, data/connectivity/raw/aconnectome_white_1986_L4.csv]",2
64,"['extension', 'filename', 'full_path', 'relative_path', 'size_kb']","[utils/final_directory_context.csv, utils/postmove_directory_inventory.csv]",2
52,"['Source', 'Target', 'Type', 'Weight']","[data/connectivity/raw/herm_full_edgelist_MODIFIED.csv, data/connectivity/raw/herm_full_edgelist.csv]",2
62,"['data_sources', 'from_neuron', 'from_pos', 'from_type', 'mean_chem_weight', 'mean_gap_weight', 'to_neuron', 'to_pos', 'to_type', 'uncertainty']","[data/connectivity/processed/consensus_connectome_full_nofunc.csv, data/connectivity/processed/consensus_connectome_trimmed_nofunc.csv]",2
61,"['data_sources', 'from_neuron', 'from_pos', 'from_type', 'functional_weight', 'mean_chem_weight', 'mean_gap_weight', 'to_neuron', 'to_pos', 'to_type', 'uncertainty']","[data/connectivity/processed/consensus_connectome_full_withfunc.csv, data/connectivity/processed/consensus_connectome_trimmed_withfunc.csv]",2
15,"['Canonical', 'Cluster', 'Neuron', 'NeuronName', 'TSNE_1', 'TSNE_2']","[symbolic_clusters_labeled_canonical.csv, symbolic_clusters_labeled_by_neuronname.csv]",2
42,"['Flag1', 'Flag2', 'Flag3', 'Index1', 'Index2', 'Morphology', 'Neuron', 'SegmentClass', '__SourceFile']",[normalized_schema8.csv],1
46,"['Neuron', 'Type']",[neuron_metadata.csv],1
45,"['Neuron', 'RichClubScore']",[richclub_scores.csv],1
44,"['Leiden', 'LeidenRole']",[cluster_roles.csv],1


In [11]:
# Example: Show files with the most common schema
most_common_group = grouped.iloc[0]["file"]
print("📂 Files with most common schema:\n")
for f in most_common_group:
    print("•", f)


📂 Files with most common schema:

• parsed_txt_all.csv


In [12]:
import os
import shutil
from pathlib import Path

# 📂 Define where files should go
file_to_folder = {
    "node2vec_embeddings.csv":               "data/embeddings/",
    "symbolic_clusters_labeled_canonical.csv": "data/clustering/",
    "symbolic_clusters_labeled_by_neuronname.csv": "data/clustering/",
    "nml_results.csv":                        "data/results/",
    "ExtraData_Sheet_Classification.csv":     "data/results/",
    "strength_metrics.csv":                   "data/metrics/",
    "richclub_scores.csv":                    "data/metrics/",
    "cluster_roles.csv":                      "data/metrics/",
    "neuron_metadata.csv":                    "data/metadata/",
    "aconnectome_white_1986_whole.csv":       "data/connectivity/raw/",
    "aconnectome_white_1986_L4.csv":          "data/connectivity/raw/",
    "herm_full_edgelist.csv":                 "data/connectivity/raw/",
    "herm_full_edgelist_MODIFIED.csv":        "data/connectivity/raw/",
    "consensus_connectome_full_nofunc.csv":   "data/connectivity/processed/",
    "consensus_connectome_trimmed_nofunc.csv": "data/connectivity/processed/",
    "consensus_connectome_full_withfunc.csv": "data/connectivity/processed/",
    "consensus_connectome_trimmed_withfunc.csv": "data/connectivity/processed/",
    "final_directory_context.csv":            "data/utils/",
    "postmove_directory_inventory.csv":       "data/utils/",
    "parsed_txt_all.csv":                     "data/legacy_or_unknown/",
}

# 📁 Ensure all destination folders exist
for folder in set(file_to_folder.values()):
    Path(folder).mkdir(parents=True, exist_ok=True)
    # Add .gitkeep to keep empty folders in version control
    with open(Path(folder) / ".gitkeep", "w") as f:
        f.write("")

# 🚚 Move files into their folders
for filename, target_folder in file_to_folder.items():
    for path in Path(".").rglob(filename):
        dest = Path(target_folder) / path.name
        if not dest.exists():
            shutil.move(str(path), str(dest))
            print(f"📦 Moved {path} → {dest}")
        else:
            print(f"⚠️ Skipped (already exists): {path}")

print("✅ CSV reorganization complete.")


📦 Moved node2vec_embeddings.csv → data/embeddings/node2vec_embeddings.csv
⚠️ Skipped (already exists): data/embeddings/node2vec_embeddings.csv
📦 Moved symbolic_clusters_labeled_canonical.csv → data/clustering/symbolic_clusters_labeled_canonical.csv
⚠️ Skipped (already exists): data/clustering/symbolic_clusters_labeled_canonical.csv
📦 Moved symbolic_clusters_labeled_by_neuronname.csv → data/clustering/symbolic_clusters_labeled_by_neuronname.csv
⚠️ Skipped (already exists): data/clustering/symbolic_clusters_labeled_by_neuronname.csv
📦 Moved nml_results.csv → data/results/nml_results.csv
⚠️ Skipped (already exists): data/results/nml_results.csv
📦 Moved ExtraData_Sheet_Classification.csv → data/results/ExtraData_Sheet_Classification.csv
⚠️ Skipped (already exists): data/results/ExtraData_Sheet_Classification.csv
📦 Moved strength_metrics.csv → data/metrics/strength_metrics.csv
⚠️ Skipped (already exists): data/metrics/strength_metrics.csv
📦 Moved richclub_scores.csv → data/metrics/richclub_

In [13]:
from pathlib import Path

def print_directory_structure(root_dir=".", show_files=True, max_depth=None):
    root = Path(root_dir).resolve()
    lines = []

    def walk(path, depth=0):
        if max_depth is not None and depth > max_depth:
            return
        indent = "    " * depth
        folder_name = path.name if depth > 0 else str(path)
        files = list(path.glob("*"))
        subdirs = [f for f in files if f.is_dir()]
        regular_files = [f for f in files if f.is_file()]

        lines.append(f"{indent}📁 {folder_name}/  ({len(regular_files)} files)")

        if show_files:
            for f in sorted(regular_files):
                lines.append(f"{indent}    📄 {f.name}")
        for d in sorted(subdirs):
            walk(d, depth + 1)

    walk(root)
    return "\n".join(lines)

# 📊 Run and output the full structure
layout_output = print_directory_structure(".", show_files=True)
print(layout_output)

# 💾 Optional: Save to file
with open("project_structure.txt", "w") as f:
    f.write(layout_output)


📁 /home/rohit/Desktop/Projects/C-Elegans/  (55 files)
    📄 .gitignore
    📄 automl_neuron.log
    📄 cleanup.ipynb
    📄 cluster_feature_summary.csv
    📄 csv_column_index.csv
    📄 datasetbuilder.ipynb
    📄 extra_file_catalog.csv
    📄 extradata_file_index.csv
    📄 kmeans_cluster_roles.csv
    📄 latent_archetype_labels.csv
    📄 lineageexploration.ipynb
    📄 motif_class_table.csv
    📄 motif_counts_with_structure.csv
    📄 motif_global_counts.csv
    📄 motif_global_counts_cleaned.csv
    📄 motif_global_counts_labeled.csv
    📄 motif_null_trials.csv
    📄 motif_participation_profiles.csv
    📄 motif_scores.csv
    📄 motif_type_anova_results.csv
    📄 motif_type_boxplot.png
    📄 nca_symbolic_clusters.csv
    📄 nca_symbolic_clusters_labeled.csv
    📄 nca_symbolic_clusters_with_names.csv
    📄 neuron_master.csv
    📄 neuron_motif_roles.csv
    📄 normalized_schema8.csv
    📄 normalized_schema9.csv
    📄 oscillator_neurons.txt
    📄 out_metadata_notes.csv
    📄 out_morphology_segments.c

In [14]:
from pathlib import Path
import shutil

# ─────────────────────────────────────────────────────────────
# 1) Define your buckets (destination folders)
# ─────────────────────────────────────────────────────────────
buckets = {
    # Code & notebooks
    "*.ipynb":   "notebooks/",
    "*.py":      "scripts/",
    # Logs, artifacts
    "*.log":     "logs/",
    "*.pdf":     "docs/",
    "*.tex":     "docs/",
    "*.txt":     "docs/",
    # Root-level CSVs (already cleaned): collect any leftovers
    "*.csv":     "data/root_csvs/",
    # Models
    "*.pt":      "models/",
    # Figures & images
    "*.png":     "figures/",
    "*.jpg":     "figures/",
    # WormBase/raw downloads
    "Downloads/":            "data/raw/wormbase_full_release/",
    "wormbase_full_release/": "data/raw/wormbase_full_release/",
    # Catch-all for HTML pages / listings
    "index.html*":           "trash/html_listings/",
}

# ─────────────────────────────────────────────────────────────
# 2) Create buckets & .gitkeep
# ─────────────────────────────────────────────────────────────
for dest in set(buckets.values()):
    dst = Path(dest)
    dst.mkdir(parents=True, exist_ok=True)
    # keep empty dirs in git
    (dst / ".gitkeep").write_text("")

# ─────────────────────────────────────────────────────────────
# 3) Move files / folders into the buckets
# ─────────────────────────────────────────────────────────────
root = Path(".").resolve()

for pattern, dest in buckets.items():
    # If pattern is a folder name, move that entire folder
    if root.joinpath(pattern).is_dir():
        src = root / pattern
        dst = root / dest / pattern
        if not dst.exists():
            shutil.move(str(src), str(dst))
            print(f"📂 Moved folder {src} → {dst}")
        continue

    # Otherwise treat it as a glob for files
    for path in root.rglob(pattern):
        # skip anything already in a bucket
        if Path(dest) in path.parents:
            continue
        dst = root / dest / path.name
        # avoid overwriting
        if dst.exists():
            print(f"⚠️ Skipped (exists): {path}")
        else:
            shutil.move(str(path), str(dst))
            print(f"📦 Moved {path} → {dst}")

# ─────────────────────────────────────────────────────────────
# 4) Final touches: collapse empty folders
# ─────────────────────────────────────────────────────────────
for d in root.rglob("*"):
    if d.is_dir() and not any(d.iterdir()):
        d.rmdir()
        print(f"🧹 Removed empty folder: {d}")

print("✅ Reorganization complete!")


📦 Moved /home/rohit/Desktop/Projects/C-Elegans/overnight2.ipynb → /home/rohit/Desktop/Projects/C-Elegans/notebooks/overnight2.ipynb
📦 Moved /home/rohit/Desktop/Projects/C-Elegans/overnight_symbolic_pipeline.ipynb → /home/rohit/Desktop/Projects/C-Elegans/notebooks/overnight_symbolic_pipeline.ipynb
📦 Moved /home/rohit/Desktop/Projects/C-Elegans/datasetbuilder.ipynb → /home/rohit/Desktop/Projects/C-Elegans/notebooks/datasetbuilder.ipynb
📦 Moved /home/rohit/Desktop/Projects/C-Elegans/phase1.ipynb → /home/rohit/Desktop/Projects/C-Elegans/notebooks/phase1.ipynb
📦 Moved /home/rohit/Desktop/Projects/C-Elegans/lineageexploration.ipynb → /home/rohit/Desktop/Projects/C-Elegans/notebooks/lineageexploration.ipynb
📦 Moved /home/rohit/Desktop/Projects/C-Elegans/phase0.ipynb → /home/rohit/Desktop/Projects/C-Elegans/notebooks/phase0.ipynb
📦 Moved /home/rohit/Desktop/Projects/C-Elegans/cleanup.ipynb → /home/rohit/Desktop/Projects/C-Elegans/notebooks/cleanup.ipynb
📦 Moved /home/rohit/Desktop/Projects/C-E

Error: Cannot move a directory '/home/rohit/Desktop/Projects/C-Elegans/data/raw/wormbase_full_release' into itself '/home/rohit/Desktop/Projects/C-Elegans/data/raw/wormbase_full_release/wormbase_full_release'.

In [15]:
print_directory_structure(".", show_files=False)

'📁 /home/rohit/Desktop/Projects/C-Elegans/  (1 files)\n    📁 .git/  (5 files)\n        📁 branches/  (0 files)\n        📁 hooks/  (13 files)\n        📁 info/  (1 files)\n        📁 logs/  (1 files)\n            📁 refs/  (0 files)\n                📁 heads/  (1 files)\n                📁 remotes/  (0 files)\n                    📁 origin/  (1 files)\n        📁 objects/  (0 files)\n            📁 00/  (3 files)\n            📁 01/  (4 files)\n            📁 02/  (3 files)\n            📁 03/  (4 files)\n            📁 04/  (2 files)\n            📁 05/  (3 files)\n            📁 06/  (3 files)\n            📁 07/  (4 files)\n            📁 09/  (1 files)\n            📁 0a/  (2 files)\n            📁 0b/  (3 files)\n            📁 0c/  (3 files)\n            📁 0e/  (3 files)\n            📁 0f/  (2 files)\n            📁 10/  (1 files)\n            📁 12/  (3 files)\n            📁 13/  (3 files)\n            📁 14/  (2 files)\n            📁 15/  (4 files)\n            📁 16/  (1 files)\n            📁 17/  (3 

In [16]:
import os

def print_tree(root, prefix=""):
    """
    Recursively prints a directory tree.

    root   : path to the directory you want to list
    prefix : used internally for drawing the tree branches
    """
    try:
        entries = sorted(os.listdir(root))
    except PermissionError:
        return

    for idx, name in enumerate(entries):
        path = os.path.join(root, name)
        # choose branch or end-of-branch connector
        connector = "└── " if idx == len(entries) - 1 else "├── "
        print(prefix + connector + name)

        if os.path.isdir(path):
            # adjust the prefix for children
            extension = "    " if idx == len(entries) - 1 else "│   "
            print_tree(path, prefix + extension)

# ── CONFIGURE HERE ───────────────────────────────────────────────────────────
PROJECT_ROOT = "/home/rohit/Desktop/C-Elegans"
# ─────────────────────────────────────────────────────────────────────────────

print(PROJECT_ROOT)
print_tree(PROJECT_ROOT)


/home/rohit/Desktop/Projects/C-Elegans
├── .git
│   ├── COMMIT_EDITMSG
│   ├── HEAD
│   ├── branches
│   ├── config
│   ├── description
│   ├── hooks
│   │   ├── applypatch-msg.sample
│   │   ├── commit-msg.sample
│   │   ├── fsmonitor-watchman.sample
│   │   ├── post-update.sample
│   │   ├── pre-applypatch.sample
│   │   ├── pre-commit.sample
│   │   ├── pre-merge-commit.sample
│   │   ├── pre-push.sample
│   │   ├── pre-rebase.sample
│   │   ├── pre-receive.sample
│   │   ├── prepare-commit-msg.sample
│   │   ├── push-to-checkout.sample
│   │   └── update.sample
│   ├── index
│   ├── info
│   │   └── exclude
│   ├── logs
│   │   ├── HEAD
│   │   └── refs
│   │       ├── heads
│   │       │   └── main
│   │       └── remotes
│   │           └── origin
│   │               └── main
│   ├── objects
│   │   ├── 00
│   │   │   ├── 7216fba880c4eaba2056e1eee03585796331ec
│   │   │   ├── 7423b3851976a6bb5287d02f50bb5534dd8294
│   │   │   └── fd5ba815576fac4ff163975f4eecddb21b6b44
│   │   ├──

In [18]:
import os

def find_empty_and_redundant_folders(root, print_empty=True, print_redundant=True, low_threshold=1):
    empty_dirs = []
    redundant_dirs = []
    low_content_dirs = []

    for dirpath, dirnames, filenames in os.walk(root, topdown=False):
        # Count only files directly in this dir
        file_count = len(filenames)
        subdir_count = len(dirnames)

        # Check if this folder is completely empty
        if file_count == 0 and subdir_count == 0:
            empty_dirs.append(dirpath)
        # Check if it contains only other empty dirs
        elif all(os.path.join(dirpath, d) in empty_dirs for d in dirnames) and file_count == 0:
            redundant_dirs.append(dirpath)
        # Optionally: flag low-content folders
        elif file_count <= low_threshold and subdir_count == 0:
            low_content_dirs.append((dirpath, file_count))

    if print_empty:
        print("\n📂 Completely Empty Folders:")
        for path in empty_dirs:
            print("  -", path)

    if print_redundant:
        print("\n🗃️ Redundant Folders (only contain empty subfolders):")
        for path in redundant_dirs:
            print("  -", path)

    print("\n⚠️ Low-content Folders (≤1 file):")
    for path, count in low_content_dirs:
        print(f"  - {path} ({count} file)")

    return empty_dirs, redundant_dirs, low_content_dirs

# ✅ Run it on your C-Elegans project folder
if __name__ == "__main__":
    PROJECT_ROOT = "/home/rohit/Desktop/C-Elegans"
    empty, redundant, low = find_empty_and_redundant_folders(PROJECT_ROOT)



📂 Completely Empty Folders:
  - /home/rohit/Desktop/Projects/C-Elegans/_extracted_txt/Ce_synapse_UNIX/Ce_synapse/Albertson_Fig/neurons
  - /home/rohit/Desktop/Projects/C-Elegans/_extracted_txt/Ce_synapse_UNIX/Ce_synapse/White_Fig/neurons
  - /home/rohit/Desktop/Projects/C-Elegans/_extracted_txt/Ce_synapse_UNIX/Ce_synapse/White_Table/neurons
  - /home/rohit/Desktop/Projects/C-Elegans/_extracted_txt/database.297.4-0/wquery/geneace
  - /home/rohit/Desktop/Projects/C-Elegans/extracted_tar_archives
  - /home/rohit/Desktop/Projects/C-Elegans/data/lineage/processed
  - /home/rohit/Desktop/Projects/C-Elegans/data/exports
  - /home/rohit/Desktop/Projects/C-Elegans/data/neuroml/circuits
  - /home/rohit/Desktop/Projects/C-Elegans/.git/objects/info
  - /home/rohit/Desktop/Projects/C-Elegans/.git/objects/pack
  - /home/rohit/Desktop/Projects/C-Elegans/.git/refs/tags
  - /home/rohit/Desktop/Projects/C-Elegans/.git/branches
  - /home/rohit/Desktop/Projects/C-Elegans/vgae_outputs/logs
  - /home/rohit

In [19]:
import os
import shutil

# Set this to the root of your project directory
PROJECT_ROOT = "."

# Folders flagged for deletion (relative to PROJECT_ROOT)
folders_to_delete = [
    "data/temp",
    "data/embeddings/old",
    "data/connectivity/temp",
    "data/legacy_or_unknown/old",
    "data/results/temp",
    "data/metrics/unused",
    "data/utils/tmp",
]

# Optional: Folders to review manually
manual_review = [
    "data/metadata/misc",
    "data/utils/notes",
]

deleted = []
skipped = []

for rel_path in folders_to_delete:
    full_path = os.path.join(PROJECT_ROOT, rel_path)
    if os.path.exists(full_path):
        try:
            shutil.rmtree(full_path)
            deleted.append(rel_path)
        except Exception as e:
            skipped.append((rel_path, str(e)))
    else:
        skipped.append((rel_path, "Path does not exist"))

# Log results
print("✅ Deleted Folders:")
for d in deleted:
    print("📦", d)

if skipped:
    print("\n⚠️ Skipped:")
    for path, reason in skipped:
        print(f"⛔ {path} — {reason}")

if manual_review:
    print("\n🕵️ Manual Review Suggested:")
    for m in manual_review:
        print("🔍", m)


✅ Deleted Folders:
📦 data/temp

⚠️ Skipped:
⛔ data/embeddings/old — Path does not exist
⛔ data/connectivity/temp — Path does not exist
⛔ data/legacy_or_unknown/old — Path does not exist
⛔ data/results/temp — Path does not exist
⛔ data/metrics/unused — Path does not exist
⛔ data/utils/tmp — Path does not exist

🕵️ Manual Review Suggested:
🔍 data/metadata/misc
🔍 data/utils/notes


In [20]:
import os

target_file = "merged_master_graph.gml"
matches = []

for root, dirs, files in os.walk("."):
    if target_file in files:
        matches.append(os.path.join(root, target_file))

if matches:
    print("✅ Found:")
    for m in matches:
        print("📍", os.path.abspath(m))
else:
    print("❌ File not found.")


❌ File not found.


In [21]:
import os

# Define target filenames to search for
target_files = {
    "motif_summary.csv": None,
    "lineage_embeddings.pkl": None,
    "richscore_df.pkl": None
}

# Walk through all files in the project directory
for root, _, files in os.walk("."):
    for file in files:
        if file in target_files:
            target_files[file] = os.path.join(root, file)

# Print results
for name, path in target_files.items():
    if path:
        print(f"✅ Found {name}: {path}")
    else:
        print(f"❌ {name} not found.")


❌ motif_summary.csv not found.
❌ lineage_embeddings.pkl not found.
❌ richscore_df.pkl not found.


In [22]:
import os

keywords = ["motif", "lineage", "richscore"]
matches = []

# Recursively search
for root, _, files in os.walk("."):
    for file in files:
        lower = file.lower()
        if any(k in lower for k in keywords):
            matches.append(os.path.join(root, file))

# Print results
if matches:
    print("🔍 Possible matches:")
    for m in matches:
        print("  •", m)
else:
    print("❌ No partial matches found for motif, lineage, or richscore.")


🔍 Possible matches:
  • ./lineageexploration.ipynb
  • ./data/lineage/raw/NeuronLineage_Part2.xls
  • ./data/lineage/raw/NeuronLineage_Part1.xls
  • ./data/root_csvs/motif_participation_profiles.csv
  • ./data/root_csvs/motif_null_trials.csv
  • ./data/root_csvs/neuron_motif_roles.csv
  • ./data/root_csvs/merged_lineage_table.csv
  • ./data/root_csvs/motif_global_counts_labeled.csv
  • ./data/root_csvs/motif_divergence.csv
  • ./data/root_csvs/motif_class_table.csv
  • ./data/root_csvs/motif_global_counts.csv
  • ./data/root_csvs/motif_global_counts_cleaned.csv
  • ./data/root_csvs/motif_enrichment.csv
  • ./data/root_csvs/top_rich_motif_neurons.csv
  • ./data/root_csvs/motif_type_anova_results.csv
  • ./data/root_csvs/motif_scores.csv
  • ./data/root_csvs/motif_counts_with_structure.csv
  • ./vgae_outputs/overnight_analysis/motif_enrichment_sankey.html
  • ./figures/motif_type_boxplot.png
  • ./figures/motif_divergence_heatmap.png
  • ./ExtraData/NeuronLineage_Part2(1).xls
  • ./Extra

In [24]:
import pandas as pd

# Load substitute files (just show shape and columns)
files = {
    "motif_participation_profiles.csv": "analysis/motif/motif_participation_profiles.csv",
    "merged_lineage_table.csv": "data/lineage/merged_lineage_table.csv",
    "top_rich_motif_neurons.csv": "analysis/motif/top_rich_motif_neurons.csv"
}

for name, path in files.items():
    try:
        df = pd.read_csv(path)
        print(f"✅ {name} — Shape: {df.shape}")
        print("📌 Columns:", df.columns.tolist())
        print()
    except Exception as e:
        print(f"❌ Failed to load {name}: {e}")


✅ motif_participation_profiles.csv — Shape: (1533, 419)
📌 Columns: ['Neuron', '(0, 1)_(0, 2)_(1, 2)_nan_nan_nan_nan_nan_nan', '(0, 1)_(0, 2)_nan_nan_nan_nan_nan_nan_nan', '(0, 1)_(2, 1)_nan_nan_nan_nan_nan_nan_nan', '(0, 1)_(0, 2)_(2, 2)_nan_nan_nan_nan_nan_nan', '(0, 1)_(0, 2)_(2, 1)_(2, 2)_nan_nan_nan_nan_nan', '(0, 1)_(2, 1)_(2, 2)_nan_nan_nan_nan_nan_nan', '(0, 1)_(2, 0)_(2, 2)_nan_nan_nan_nan_nan_nan', '(0, 2)_(1, 1)_(1, 2)_nan_nan_nan_nan_nan_nan', '(0, 2)_(1, 2)_nan_nan_nan_nan_nan_nan_nan', '(0, 2)_(1, 0)_(1, 1)_nan_nan_nan_nan_nan_nan', '(0, 1)_(0, 2)_(2, 1)_nan_nan_nan_nan_nan_nan', '(0, 1)_(2, 0)_nan_nan_nan_nan_nan_nan_nan', '(0, 1)_(0, 2)_(2, 0)_nan_nan_nan_nan_nan_nan', '(0, 1)_(0, 2)_(1, 2)_(2, 2)_nan_nan_nan_nan_nan', '(0, 1)_(1, 2)_nan_nan_nan_nan_nan_nan_nan', '(0, 1)_(1, 2)_(2, 0)_(2, 2)_nan_nan_nan_nan_nan', '(0, 1)_(2, 0)_(2, 1)_(2, 2)_nan_nan_nan_nan_nan', '(0, 1)_(1, 2)_(2, 2)_nan_nan_nan_nan_nan_nan', '(0, 1)_(1, 2)_(2, 1)_nan_nan_nan_nan_nan_nan', '(0, 1)_(0, 2

In [28]:
import numpy as np
import logging

# Optional: set up basic logging to cell output
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')


In [31]:
import pandas as pd
import networkx as nx
import numpy as np
import logging
import os

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# Paths
# Paths to your uploaded files
edge_path = 'merged_edges_with_relatedness.csv'
meta_path = 'neuron_metadata.csv'  # or neuron_master.csv
motif_path = 'motif_participation_profiles.csv'
lineage_path = 'merged_lineage_table.csv'
richscore_path = 'top_rich_motif_neurons.csv'
output_path = 'output/merged_master_graph.gml'

# Load data
edges = pd.read_csv(edge_path)
metadata = pd.read_csv(meta_path, index_col=0)
motifs = pd.read_csv(motif_path, index_col=0)
lineage = pd.read_csv(lineage_path, index_col=0)
richscore = pd.read_csv(richscore_path, index_col=0)

# Normalize indices to strings
motifs.index = motifs.index.astype(str)
metadata.index = metadata.index.astype(str)
lineage.index = lineage.index.astype(str)
richscore.index = richscore.index.astype(str)

# Optional: key sanitizer for GML
def sanitize_gml_key(s):
    return str(s).replace("(", "").replace(")", "").replace(",", "_").replace(" ", "").replace("-", "_")

# Build graph
G = nx.DiGraph()
all_neurons = set(metadata.index) | set(motifs.index) | set(lineage.index) | set(richscore.index)

for neuron in all_neurons:
    attrs = {}

    # Basic metadata
    if neuron in metadata.index:
        attrs.update(metadata.loc[neuron].to_dict())

    # Motif participation (sanitize keys)
    if neuron in motifs.index:
        for k, v in motifs.loc[neuron].items():
            safe_k = f"motif_{sanitize_gml_key(k)}"
            try:
                attrs[safe_k] = float(v)
            except:
                logging.warning(f"Could not convert motif {k} value {v} for neuron {neuron}")

    # Lineage values (extract only numeric similarity score)
    if neuron in lineage.index:
        lineage_vals = lineage.loc[neuron].values
        for i, val in enumerate(lineage_vals):
            try:
                if isinstance(val, (list, np.ndarray)) and len(val) == 2:
                    val = float(val[1])
                else:
                    val = float(val)
                attrs[f'lineage_{i}'] = val
            except Exception as e:
                logging.warning(f"Failed to parse lineage value for {neuron} at index {i}: {val} ({e})")

    # RichScore
    if neuron in richscore.index:
        val = richscore.loc[neuron].values
        try:
            attrs['RichScore'] = float(val[0]) if len(val) == 1 else float(val[-1])
        except:
            logging.warning(f"Could not convert RichScore for {neuron}")

    G.add_node(neuron, **attrs)

# Add edges
for _, row in edges.iterrows():
    src, tgt = str(row['Source']), str(row['Target'])
    if src in G and tgt in G:
        G.add_edge(src, tgt,
                   Type=row.get('Type', 'chemical'),
                   SynapseCount=row.get('SynapseCount', 1),
                   Relatedness=row.get('Relatedness', None))

# Save
nx.write_gml(G, output_path)
print(f"✅ Saved merged_master_graph.gml with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")


✅ Saved merged_master_graph.gml with 1633 nodes and 10902 edges.


In [32]:
import networkx as nx
import pandas as pd
import pickle

# Load the graph
G = nx.read_gml("output/merged_master_graph.gml")

# Extract node data
node_df = pd.DataFrame.from_dict(dict(G.nodes(data=True)), orient="index")

# Extract motif features
motif_cols = [col for col in node_df.columns if col.startswith("motif_")]
motif_summary = node_df[motif_cols].copy()
motif_summary.fillna(0, inplace=True)
motif_summary.to_csv("motif_summary.csv")
print(f"✅ Rebuilt motif_summary.csv with shape {motif_summary.shape}")

# Extract lineage features
lineage_cols = [col for col in node_df.columns if col.startswith("lineage_")]
lineage_embeddings = node_df[lineage_cols].copy()
lineage_embeddings.fillna(0, inplace=True)
with open("lineage_embeddings.pkl", "wb") as f:
    pickle.dump(lineage_embeddings, f)
print(f"✅ Rebuilt lineage_embeddings.pkl with shape {lineage_embeddings.shape}")


✅ Rebuilt motif_summary.csv with shape (1633, 418)
✅ Rebuilt lineage_embeddings.pkl with shape (1633, 278)
